# The goal 
The goal of this notebook is to try classic PPO on a simple game RugbyHRL designed to be used with variante and understand task scheduling

https://medium.com/@coldstart_coder/ppo-pytorch-implementation-4322372a91d2

In [18]:
#Imports
import sys
import os
sys.path.append(os.path.abspath('..')) 
import numpy as np
import random
import torch
import gymnasium as gym
import src.env

## Config
Here we will choose our env variable   

In [ ]:
seed = 444
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True

env_name = "RugbyHRLGame1-v0"
num_envs=10
envs = gym.make_vec(env_name, num_envs=num_envs, vectorization_mode="sync")

/home/wboussella/Documents/RugbyHRL/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [20]:
from tensordict import TensorDict
import torch

class PPORolloutCollector:
    def __init__(self, agent, envs, num_steps_per_rollout, device, eval_env=None):
        self.agent = agent
        self.envs = envs
        self.num_envs = envs.num_envs
        self.num_steps_per_rollout = num_steps_per_rollout
        self.device = device
        self.eval_env = eval_env

        self.obs_shape = envs.single_observation_space.shape
        self.action_shape = envs.single_action_space.shape
        self.initial_buffer_shape = (num_steps_per_rollout, envs.num_envs)

        # go ahead and reset the environment and store the observation
        # and set next_done to false (we assume the environment can't start terminal)
        obs, _ = envs.reset()
        self.next_obs = torch.Tensor(obs).to(device)
        self.next_done = torch.zeros(num_envs).to(device)

    def _create_buffer(self):
        return TensorDict({
            "obs": torch.zeros(self.initial_buffer_shape + self.obs_shape).to(self.device),
            "actions": torch.zeros(self.initial_buffer_shape + self.action_shape).to(self.device),
            "log_probs": torch.zeros(self.initial_buffer_shape).to(self.device),
            "rewards": torch.zeros(self.initial_buffer_shape).to(self.device),
            "dones": torch.zeros(self.initial_buffer_shape).to(self.device),
            "critic_values": torch.zeros(self.initial_buffer_shape).to(self.device),
        })

    # function that will collect num_steps_per_rollout observations against our training environment
    def get_next_rollout(self):
        buffer = self._create_buffer()

        # get the last recorded observation and if that observation is terminal
        next_obs = self.next_obs
        next_done = self.next_done

        # collect the rollouts,
        for t in range(self.num_steps_per_rollout):
            # record the current observation and terminal
            buffer["obs"][t] = next_obs
            buffer["dones"][t] = next_done

            # query the agent for the next action, the log prob of that action, and the critic estimation
            with torch.no_grad():
                action, log_prob, entropy = self.agent.get_actor_values(next_obs)
                critic_value = self.agent.get_critic_value(next_obs)

            # record the values
            buffer["actions"][t] = action
            buffer["log_probs"][t] = log_prob
            buffer["critic_values"][t] = critic_value.flatten()

            # perform the action
            next_obs, reward, terminations, truncations, infos = envs.step(action.cpu().numpy())

            # shape and store the rewards
            reward = torch.tensor(reward).to(self.device).view(-1)
            buffer["rewards"][t] = reward

            # some environments will terminate (meaning the agent is in a final state),
            # others will truncate (ex a time limit is reached but not in a terminal state)
            # these are important distinctions, but for our purposes mean the same thing, the simulation ended.
            # so if either is true and the simulation resets we set next done to true
            next_done = np.logical_or(terminations, truncations)
            # store the next obs and next done for the next round
            next_obs, next_done = torch.Tensor(next_obs).to(self.device), torch.Tensor(next_done).to(self.device)

        # store the next obs and next done in the buffer
        # this is to handle edge cases when calculating advantages later,
        buffer['next_obs'] = next_obs
        buffer['next_done'] = next_done
        # we'll also need the critic estimate for this next state, we'll use this to bootstrap the reward for the final state
        # when we calculate gae
        with torch.no_grad():
            buffer['next_value'] = self.agent.get_critic_value(next_obs).reshape(1, -1)
        self.next_obs = next_obs
        self.next_done = next_done
        return buffer


    # meant for evaluation on the agent on our environment, and will run an entire simulation to termination
    # very similar to the above, the only difference is that we will manually check if the environment terminates
    # and then end the loop,
    # will return a normal python dict with the rewards, entropy, reward averages, average entropy of our agent, and the total rewards for each run
    def run_eval_rollout(self, num_episodes: int = 5):
        assert self.eval_env is not None, "No eval_env provided."

        rewards_per_timestep = []
        entropies_per_timestep = []
        final_rewards = []
        total_entropies = []

        for _ in range(num_episodes):
            obs, _ = self.eval_env.reset()
            obs = torch.tensor(obs, device=self.device).unsqueeze(0)
            done = False

            episode_rewards = []
            episode_entropies = []

            while not done:
                with torch.no_grad():
                    action, _, entropy = self.agent.get_actor_values(obs)
                    action = action.squeeze()
                obs_np, reward, term, trunc, _ = self.eval_env.step(action.cpu().numpy())
                done = term or trunc

                obs = torch.tensor(obs_np, device=self.device).unsqueeze(0)
                episode_rewards.append(float(reward))
                episode_entropies.append(entropy.item())

            rewards_per_timestep.append(episode_rewards)
            entropies_per_timestep.append(episode_entropies)
            final_rewards.append(sum(episode_rewards))
            total_entropies.append(sum(episode_entropies) / len(episode_entropies))

        return {
            "rewards_per_timestep": rewards_per_timestep,
            "average_reward_per_run": sum(final_rewards) / len(final_rewards),
            "average_entropy_per_run": sum(total_entropies) / len(total_entropies),
            "entropies_per_timestep": entropies_per_timestep,
            "final_rewards": final_rewards,
        }